In [13]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

print("Starting Silver → Gold Transformation...")

# Configuration
SILVER_TABLE = "silver.customerchurn"
GOLD_TABLE = "gold.customer_summary"
BATCH_SIZE = 1000

  # Step 1: Read silver data (incremental)
df_silver = spark.sql(f"""SELECT * FROM {SILVER_TABLE} """)
display(df_silver)

StatementMeta(, 6021df36-ec39-4acc-a6f1-29869f8f0b86, 15, Finished, Available, Finished, False)

Starting Silver → Gold Transformation...


SynapseWidget(Synapse.DataFrame, ab479fba-c2f7-458c-9c0d-da43f28505a7)

In [15]:
df_silver = df_silver.drop_duplicates()

StatementMeta(, 6021df36-ec39-4acc-a6f1-29869f8f0b86, 17, Finished, Available, Finished, False)

In [16]:
row_count_before = df_silver.count()
print(f"Processing {row_count_before} records from silver layer...")
    
if row_count_before == 0:
    print("⚠ No new data to process. Exiting.")
    dbutils.notebook.exit("NO_DATA")

StatementMeta(, 6021df36-ec39-4acc-a6f1-29869f8f0b86, 18, Finished, Available, Finished, False)

Processing 5005 records from silver layer...


In [17]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

print("Starting Silver → Gold Transformation...")

# Configuration
SILVER_TABLE = "silver.customerchurn"
BATCH_SIZE = 10000

  # Step 1: Read silver data (incremental)
df_silver = spark.sql(f"""SELECT * FROM {SILVER_TABLE}""")
display(df_silver)

row_count_before = df_silver.count()
print(f"Processing {row_count_before} records from silver layer...")
    
if row_count_before == 0:
    print("⚠ No new data to process. Exiting.")
    dbutils.notebook.exit("NO_DATA")

#  # Step 2: Aggregate data

df_gold = (df_silver
    .orderBy(
        col("cr8b0_internetservicetype")
    )
    .groupBy(
        col("cr8b0_internetservicetype"),
        col("cr8b0_paymentmethod")
    )
    .agg(
        count("*").alias("transaction_count"),
        sum(col("cr8b0_totalcharges")).alias("TotalSumCharges")
    )
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("load_date", current_date())
)

row_count_gold = df_gold.count()
print(f"Aggregated to {row_count_gold} gold records")


# Step 3: Write to gold (append, partitioned by load_date)
df_gold.write.format("delta").mode("overwrite").saveAsTable("gold.CustomerChurn")

display(df_gold)

StatementMeta(, 6021df36-ec39-4acc-a6f1-29869f8f0b86, 19, Finished, Available, Finished, False)

Starting Silver → Gold Transformation...


SynapseWidget(Synapse.DataFrame, 8068fc63-61fc-4586-9899-3caacf7b8e68)

Processing 15015 records from silver layer...
Aggregated to 12 gold records


SynapseWidget(Synapse.DataFrame, d0c9a4b5-b6ed-4e21-960e-8237eb50a837)